
# 02a_Metadaten_Features_Engineering

Dieses Notebook erzeugt **deterministische Features** aus der validierten Basistabelle `metadata_base.(parquet|csv)`.
Keine leaky Spalten, volle Reproduzierbarkeit.


## 1) Setup & Laden

In [1]:

from pathlib import Path
import pandas as pd, numpy as np, re, json, warnings
warnings.filterwarnings("ignore")

def read_parquet_or_csv(base_name: str):
    p_parquet = Path(f"{base_name}.parquet")
    p_csv = Path(f"{base_name}.csv")
    if p_parquet.exists():
        df = pd.read_parquet(p_parquet); print("Geladen (Parquet):", p_parquet.resolve()); return df
    elif p_csv.exists():
        df = pd.read_csv(p_csv); print("Geladen (CSV):", p_csv.resolve()); return df
    else:
        raise FileNotFoundError(f"Weder {p_parquet} noch {p_csv} gefunden. Bitte 01b zuerst ausführen.")

base = read_parquet_or_csv("metadata_base")
print("Shape:", base.shape)
display(base.head(5))

LEAKY = {"likes","views","comments","shares","engagement_score","rank"}
present_leaky = [c for c in base.columns if c in LEAKY]
print("⚠️ Ignoriere (leaky):", present_leaky)


Geladen (CSV): C:\Users\lremm\OneDrive\Desktop\FHdW\Fachsemester 5\F5 PALG\Viralitaetsanalyse\Metadaten_Analyse\metadata_base.csv
Shape: (200, 13)


,rank,group,video_id,uploader,likes,comments,views,shares,engagement_score,title,duration_s,source_file,is_viral_proxy
0,10,top,7219026508239686954,6751051329931609094,2000000,7103,18000000,31700,2014206,Hype House is going through some changes. Than...,8,C:\Users\lremm\OneDrive\Desktop\FHdW\Fachsemes...,1
1,19,top,7231352152743152942,6751051329931609094,1500000,1492,15300000,18500,1502984,I somehow got unbreakable hangers,28,C:\Users\lremm\OneDrive\Desktop\FHdW\Fachsemes...,1
2,89,normal,7236485370542624042,6751051329931609094,87600,213,4000000,340,88026,Rate our mini house #MakingMyWay,18,C:\Users\lremm\OneDrive\Desktop\FHdW\Fachsemes...,0
3,22,normal,7239370205254929710,6751051329931609094,47900,222,3700000,181,48344,#ad Are you team chipIN or team chipOUT? @lays...,38,C:\Users\lremm\OneDrive\Desktop\FHdW\Fachsemes...,0
4,44,normal,7303636279936322862,6751051329931609094,59100,214,8300000,155,59528,How would you do gift wrapping without seeing?...,61,C:\Users\lremm\OneDrive\Desktop\FHdW\Fachsemes...,0


⚠️ Ignoriere (leaky): ['rank', 'likes', 'comments', 'views', 'shares', 'engagement_score']


## 2) Account-Features

In [2]:

df = base.copy()

def safe_str(s): 
    return s if isinstance(s, str) else ""

# uploader-basierte Features
if "uploader" in df.columns:
    up = df["uploader"].map(safe_str)
    df["acct_uploader_len"] = up.str.len()
    df["acct_uploader_digits_ratio"] = up.str.count(r"\d") / df["acct_uploader_len"].replace({0:np.nan})
    df["acct_uploader_digits_ratio"] = df["acct_uploader_digits_ratio"].fillna(0.0)
else:
    df["acct_uploader_len"] = np.nan
    df["acct_uploader_digits_ratio"] = np.nan

if "creator_verified" in df.columns:
    df["acct_creator_verified"] = df["creator_verified"].fillna(False).astype(int)
else:
    df["acct_creator_verified"] = 0

for c in ["creator_follower_count","creator_posts_count"]:
    newc = {"creator_follower_count":"acct_creator_follower_log",
            "creator_posts_count":"acct_creator_posts_log"}[c]
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
        df[newc] = np.log1p(df[c])
    else:
        df[newc] = np.nan


## 3) Text-/Titel-Features

In [3]:

if "title" in df.columns:
    t = df["title"].map(safe_str)
    df["txt_title_len_chars"] = t.str.len()
    df["txt_title_len_words"] = t.str.split().map(len)
    df["txt_title_has_question"] = t.str.contains(r"\?", regex=True).fillna(False).astype(int)
    df["txt_title_has_exclaim"] = t.str.contains(r"!", regex=True).fillna(False).astype(int)
    df["txt_title_has_hashtag"] = t.str.contains(r"#\w+", regex=True).fillna(False).astype(int)
    df["txt_title_has_mention"] = t.str.contains(r"@\w+", regex=True).fillna(False).astype(int)
    df["txt_title_has_url"] = t.str.contains(r"(https?://|www\.)", regex=True, case=False).fillna(False).astype(int)
else:
    df["txt_title_len_chars"] = np.nan
    df["txt_title_len_words"] = np.nan
    df["txt_title_has_question"] = 0
    df["txt_title_has_exclaim"] = 0
    df["txt_title_has_hashtag"] = 0
    df["txt_title_has_mention"] = 0
    df["txt_title_has_url"] = 0


## 4) Längen-/Struktur-Features

In [4]:

if "duration_s" in df.columns:
    df["len_duration_s"] = pd.to_numeric(df["duration_s"], errors="coerce")
    df["len_bucket"] = pd.cut(
        df["len_duration_s"],
        bins=[-np.inf, 6, 15, 30, 60, np.inf],
        labels=["very_short","short","mid","long","very_long"]
    )
else:
    df["len_duration_s"] = np.nan
    df["len_bucket"] = pd.Series(pd.Categorical([np.nan]*len(df), categories=["very_short","short","mid","long","very_long"]))


## 5) Feature-Tabelle & QC

In [5]:

LEAKY = {"likes","views","comments","shares","engagement_score","rank"}
KEY_COLS = [c for c in ["video_id","group"] if c in df.columns]
LABEL_COLS = [c for c in ["is_viral","is_viral_proxy"] if c in df.columns]
feature_cols = [c for c in df.columns if re.match(r"^(acct|txt|len|time)_", c)]
feature_cols = [c for c in feature_cols if c not in LEAKY]
feat = df[KEY_COLS + LABEL_COLS + feature_cols].copy()

miss = feat.isna().mean().sort_values(ascending=False)
print("Top 20 Missingness (Features):")
display(miss.head(20))

schema = {c: str(feat[c].dtype) for c in feat.columns}
Path("schema_metadata_features.json").write_text(json.dumps(schema, indent=2, ensure_ascii=False), encoding="utf-8")
print("Schema gespeichert:", Path("schema_metadata_features.json").resolve())

display(feat.head(5))


Top 20 Missingness (Features):


acct_creator_posts_log        1.0
acct_creator_follower_log     1.0
video_id                      0.0
group                         0.0
is_viral_proxy                0.0
acct_uploader_digits_ratio    0.0
acct_uploader_len             0.0
acct_creator_verified         0.0
txt_title_len_chars           0.0
txt_title_len_words           0.0
txt_title_has_question        0.0
txt_title_has_exclaim         0.0
txt_title_has_hashtag         0.0
txt_title_has_mention         0.0
txt_title_has_url             0.0
len_duration_s                0.0
len_bucket                    0.0
dtype: float64

Schema gespeichert: C:\Users\lremm\OneDrive\Desktop\FHdW\Fachsemester 5\F5 PALG\Viralitaetsanalyse\Metadaten_Analyse\schema_metadata_features.json


,video_id,group,is_viral_proxy,acct_uploader_len,acct_uploader_digits_ratio,acct_creator_verified,acct_creator_follower_log,acct_creator_posts_log,txt_title_len_chars,txt_title_len_words,txt_title_has_question,txt_title_has_exclaim,txt_title_has_hashtag,txt_title_has_mention,txt_title_has_url,len_duration_s,len_bucket
0,7219026508239686954,top,1,0,0.0,0,NaN,NaN,72,12,0,0,0,0,0,8,short
1,7231352152743152942,top,1,0,0.0,0,NaN,NaN,33,5,0,0,0,0,0,28,mid
2,7236485370542624042,normal,0,0,0.0,0,NaN,NaN,32,5,0,0,1,0,0,18,mid
3,7239370205254929710,normal,0,0,0.0,0,NaN,NaN,72,11,1,0,1,1,0,38,long
4,7303636279936322862,normal,0,0,0.0,0,NaN,NaN,72,11,1,0,0,0,0,61,very_long


## 6) Export (robust)

In [ ]:

def save_parquet_or_csv(df: pd.DataFrame, base_name: str):
    base = Path(base_name)
    pq = base.with_suffix(".parquet")
    cs = base.with_suffix(".csv")
    try:
        df.to_parquet(pq, index=False)
        print("Gespeichert (Parquet):", pq.resolve())
        return pq
    except Exception as e:
        print("Parquet-Export nicht verfügbar. Fallback → CSV.", repr(e))
        df.to_csv(cs, index=False)
        print("Gespeichert (CSV):", cs.resolve())
        return cs

out_path = save_parquet_or_csv(feat, "metadata_features_raw")
out_path


ℹ️ Parquet-Export nicht verfügbar. Fallback → CSV. ImportError("Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.\nA suitable version of pyarrow or fastparquet is required for parquet support.\nTrying to import the above resulted in these errors:\n - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.\n - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.")
Gespeichert (CSV): C:\Users\lremm\OneDrive\Desktop\FHdW\Fachsemester 5\F5 PALG\Viralitaetsanalyse\Metadaten_Analyse\metadata_features_raw.csv


WindowsPath('metadata_features_raw.csv')

---

**Weiter mit 02b_Metadaten_Features_Modellierung** (Permutation Importance, finale `features/metadata_features.csv`).